# Phase 2 -- Voice Engine (EchoAuth core): LFCC + LCNN spoof detection on ASVspoof 2019 LA

Implements `docs/PHASE_2_VOICE_ENGINE.md`. Generated from `notebooks/src/03_voice_baseline.py` by
`scripts/build_notebooks.py` -- edit the source, not this .ipynb.

1. **Data + EDA** -- stream a subset of ASVspoof 2019 LA, report class balance, durations, sample rate.
2. **Preprocessing** -- resample -> silence trim -> fixed length -> LFCC + deltas, as a reusable
   pipeline used identically for train, test and the out-of-domain clips.
3. **Silence-leakage check** -- can clip timing alone predict the label? Measured before and after
   trimming, and confirmed by training the same model on untrimmed features.
4. **LCNN** -- one architecture, trained properly, class-weighted for the bonafide/spoof imbalance.
5. **Evaluation** -- EER (interpolated at the FAR/FRR crossing), DET curve, per-class accuracy.
6. **Cross-dataset spot-check** -- a few In-the-Wild clips the model never saw (stretch goal).

**Two honesty notes that shape this notebook, both explained in `docs/design_decisions.md`:**
- The loadable HuggingFace copies of ASVspoof 2019 LA contain **only the evaluation partition**, so
  the official protocol (train on attacks A01-A06, test on unseen A07-A19) is not reachable. We
  split the eval partition ourselves; attack types therefore appear on both sides, which is
  optimistic. Unseen-attack generalization is **not** measured here.
- A random clip-level split would put the same speaker in train and test. The headline split is
  therefore **speaker-disjoint**, and the random-split number is reported alongside it as a
  measurement of that leak -- the audio counterpart of Phase 1's temporal-vs-random gap.

**Kaggle:** Accelerator = GPU, Internet = on, Run All, then **Save Version -> Save & Run All (Commit)**
or the outputs will not persist.

## Setup

In [ ]:
# Kaggle: Accelerator = GPU, Internet = on.
import os
import subprocess
import sys

# True only when tests/smoke_notebook_03.py runs this notebook's code on synthetic audio.
SMOKE_TEST = os.environ.get("TRUSTMESH_SMOKE") == "1"


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if not SMOKE_TEST:
    # `datasets` streams the audio; `soundfile` decodes the FLAC bytes. Everything else (LFCC, LCNN)
    # is plain torch, so there is no torchaudio/librosa version coupling to worry about.
    for module_name, package in [("datasets", "datasets"), ("soundfile", "soundfile")]:
        try:
            __import__(module_name)
        except ImportError:
            pip_install(package)

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path(os.environ.get("TRUSTMESH_OUTPUT_DIR", "./artifacts/voice"))
DOCS_DIR = Path(os.environ.get("TRUSTMESH_DOCS_DIR", "./docs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE_TEST else "cpu")

# How many clips to stream out of the 71,237-clip LA evaluation partition. This is the number that
# must be quoted whenever a result from this notebook is quoted -- "started small" must not quietly
# become the final claim. ~6000 clips is roughly 370 MB of the 4.39 GB parquet.
N_CLIPS = 120 if SMOKE_TEST else 6000
CLIP_SECONDS = 4.0
TEST_FRACTION = 0.3
VAL_FRACTION = 0.2          # of the train portion
EPOCHS = 3 if SMOKE_TEST else 30
BATCH_SIZE = 8 if SMOKE_TEST else 32
LEARNING_RATE = 3e-4
PATIENCE = 6
N_OOD_CLIPS = 4 if SMOKE_TEST else 10
N_BOOTSTRAP = 50 if SMOKE_TEST else 1000  # stratified resamples for every EER comparison

RESULTS_PATH = OUTPUT_DIR / "results.json"
RESULTS = {"smoke_test": SMOKE_TEST}


def save_progress(key, value):
    RESULTS[key] = value
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")
    print(f"[progress] saved '{key}' -> {RESULTS_PATH}")


print("device:", DEVICE, "| SMOKE_TEST:", SMOKE_TEST, "| torch", torch.__version__)

## Shared voice-engine code (inlined from `engines/voice/`)

In [ ]:
# ---- inlined from engines/voice/data.py (edit the module, then rebuild) ----
"""ASVspoof 2019 LA loading and train/test splits.

Dataset: `SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` on HuggingFace -- parquet, no loading script,
71,237 clips, 16 kHz FLAC. IMPORTANT: it contains only the **evaluation** partition of LA
(7,355 bonafide / 63,882 spoof). The HF copy carrying the official train/dev/eval partitions
(`LanceaKing/asvspoof2019`) needs `trust_remote_code`, which current `datasets` versions no longer
support, so the official protocol (train on attacks A01-A06, test on unseen A07-A19) is not
reachable here. We therefore split the eval partition ourselves; see `speaker_disjoint_split` for
why the split is speaker-disjoint and what that does and does not fix.

Clips are streamed (not downloaded in full) so the subset size is a knob, and audio is fetched
undecoded and decoded here with soundfile -- decoding through the `datasets` Audio feature changed
backends across versions, and doing it ourselves keeps this working either way.
"""
import io
import json

import numpy as np

ASVSPOOF_DATASET = "SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA"
ASVSPOOF_SPLIT = "test"  # the dataset's only split: the LA evaluation partition
IN_THE_WILD_DATASET = "mueller91/In-The-Wild"
TARGET_SR = 16000
BONAFIDE, SPOOF = 1, 0  # label 1 = bonafide = the positive/target class for EER


def _decode_audio(blob):
    """Decode an undecoded `datasets` audio value to (float32 mono waveform, sample rate)."""
    import soundfile as sf

    if isinstance(blob, dict) and blob.get("bytes"):
        wave, sr = sf.read(io.BytesIO(blob["bytes"]), dtype="float32", always_2d=False)
    elif isinstance(blob, dict) and blob.get("array") is not None:
        wave, sr = np.asarray(blob["array"], dtype=np.float32), int(blob["sampling_rate"])
    else:
        path = blob["path"] if isinstance(blob, dict) else blob
        if isinstance(path, str) and "://" in path:
            # Archive-backed datasets (e.g. In-the-Wild, shipped as one zip) come back as chained fsspec
            # URLs like "zip://a.wav::hf://datasets/.../x.zip", which libsndfile can't open. fsspec can,
            # and it only range-reads the needed zip member, not the whole archive.
            import fsspec

            with fsspec.open(path, "rb") as handle:
                wave, sr = sf.read(io.BytesIO(handle.read()), dtype="float32", always_2d=False)
        else:
            wave, sr = sf.read(path, dtype="float32", always_2d=False)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    return np.ascontiguousarray(wave, dtype=np.float32), int(sr)


def load_asvspoof_subset(n_clips, seed=42, shuffle_buffer=20000, dataset=ASVSPOOF_DATASET,
                         split=ASVSPOOF_SPLIT, progress_every=500):
    """Stream a shuffled subset of ASVspoof 2019 LA. Returns a list of clip dicts."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split=split, streaming=True)
    stream = stream.cast_column("audio", Audio(decode=False))
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row["audio"])
        try:
            notes = json.loads(row.get("notes") or "{}")
        except (json.JSONDecodeError, TypeError):
            notes = {}
        # The dataset's ClassLabel is 0=bonafide, 1=spoof; we flip so 1 = bonafide = target class.
        clips.append({
            "waveform": wave,
            "sample_rate": sr,
            "label": BONAFIDE if int(row["label"]) == 0 else SPOOF,
            "speaker_id": notes.get("speaker_id") or "unknown",
            "utterance_id": notes.get("utterance_id") or row.get("path") or f"clip_{len(clips)}",
            "path": row.get("path"),
        })
        if progress_every and len(clips) % progress_every == 0:
            print(f"  loaded {len(clips)}/{n_clips} clips")
    return clips


def load_in_the_wild_clips(n_clips, seed=42, shuffle_buffer=2000, dataset=IN_THE_WILD_DATASET):
    """A handful of out-of-domain clips for the generalization spot-check. Column names on this
    dataset aren't documented, so the audio/label columns are detected rather than assumed."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split="train", streaming=True)
    columns = list(next(iter(stream.take(1))).keys())
    audio_col = next((c for c in columns if "audio" in c.lower() or c.lower() in ("file", "path")), None)
    label_col = next((c for c in columns if c.lower() in ("label", "key", "class", "type", "target")), None)
    if audio_col is None:
        raise RuntimeError(f"no audio-looking column in {columns}")

    try:
        stream = stream.cast_column(audio_col, Audio(decode=False))
    except (ValueError, TypeError):
        pass
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row[audio_col])
        raw_label = row.get(label_col)
        label = None
        if isinstance(raw_label, str):
            label = BONAFIDE if raw_label.strip().lower() in ("bona-fide", "bonafide", "real", "genuine") else SPOOF
        elif isinstance(raw_label, (int, np.integer)):
            label = int(raw_label)  # unknown polarity; recorded but flagged in the results
        clips.append({"waveform": wave, "sample_rate": sr, "label": label,
                      "raw_label": raw_label, "label_column": label_col, "speaker_id": "unknown",
                      "utterance_id": f"itw_{len(clips)}", "path": row.get("path")})
    return clips


ASVSPOOF_PROTOCOL = "protocols/ASVspoof2019.LA.cm.eval.trl.txt"
IN_THE_WILD_ARCHIVE = "datasets/mueller91/In-The-Wild/release_in_the_wild.zip"
_BONAFIDE_WORDS = {"bona-fide", "bonafide", "bona fide", "real", "genuine"}
_SPOOF_WORDS = {"spoof", "fake", "deepfake", "synthetic"}


def load_asvspoof_table(dataset=ASVSPOOF_DATASET, split=ASVSPOOF_SPLIT, cache_dir=None):
    """Download the whole parquet once and return it with audio left undecoded.

    Phase 2 streamed a shuffled sample. Phase 2B keeps *every* bonafide clip, which means touching
    every row anyway, and random access by index into a local memory-mapped table beats a streamed
    shuffle buffer for that (streaming, not training, was the slow part of Phase 2).
    """
    from datasets import Audio, load_dataset

    table = load_dataset(dataset, split=split, cache_dir=cache_dir)
    return table.cast_column("audio", Audio(decode=False))


def table_metadata(table):
    """Labels, utterance and speaker IDs for every row, without decoding any audio."""
    labels = np.array([BONAFIDE if int(v) == 0 else SPOOF for v in table["label"]])
    notes = []
    for raw in table["notes"]:
        try:
            notes.append(json.loads(raw or "{}"))
        except (json.JSONDecodeError, TypeError):
            notes.append({})
    paths = list(table["path"])
    utterances = [n.get("utterance_id") or (p.rsplit(".", 1)[0] if p else f"row_{i}")
                  for i, (n, p) in enumerate(zip(notes, paths))]
    return {
        "label": labels,
        "utterance_id": utterances,
        "speaker_id": [n.get("speaker_id") or "unknown" for n in notes],
        "columns": list(table.features),
        "note_keys": sorted({k for n in notes for k in n}),
    }


def decode_row(table, index):
    return _decode_audio(table[int(index)]["audio"])


def load_eval_protocol(path=None, dataset=ASVSPOOF_DATASET, filename=ASVSPOOF_PROTOCOL):
    """Official ASVspoof CM protocol: one line per clip, `SPEAKER UTTERANCE - ATTACK KEY`.

    The parquet has no attack column, but the same HF repo ships this file, which is the source of
    attack IDs (A07-A19). It is only trusted after `verify_protocol` passes.
    """
    if path is None:
        from huggingface_hub import hf_hub_download

        path = hf_hub_download(repo_id=dataset, filename=filename, repo_type="dataset")
    protocol = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            parts = line.split()
            if len(parts) != 5:
                continue
            speaker, utterance, _, attack, key = parts
            protocol[utterance] = {
                "speaker_id": speaker,
                "attack": "bonafide" if attack == "-" else attack,
                "label": BONAFIDE if key == "bonafide" else SPOOF,
            }
    return protocol, str(path)


def verify_protocol(protocol, utterance_ids, labels, speaker_ids):
    """The join is only usable if it reproduces the dataset's own labels and speakers exactly --
    a protocol file for a different partition, or misaligned IDs, fails this loudly."""
    n = len(utterance_ids)
    found = label_ok = speaker_ok = 0
    bonafide_clean = True
    for utterance, label, speaker in zip(utterance_ids, labels, speaker_ids):
        entry = protocol.get(utterance)
        if entry is None:
            continue
        found += 1
        label_ok += entry["label"] == label
        speaker_ok += entry["speaker_id"] == speaker
        if label == BONAFIDE and entry["attack"] != "bonafide":
            bonafide_clean = False
    return {
        "clips": n,
        "found_in_protocol": found,
        "label_agreement": label_ok,
        "speaker_agreement": speaker_ok,
        "bonafide_rows_have_no_attack": bonafide_clean,
        "usable": found == n and label_ok == n and speaker_ok == n and bonafide_clean,
    }


def select_subset(labels, spoof_per_bonafide, seed=42):
    """Every bonafide clip plus a random sample of spoof clips capped at `spoof_per_bonafide` per
    bonafide clip. If a budget forces a smaller set, lower the ratio -- bonafide is never cut."""
    labels = np.asarray(labels)
    bonafide = np.flatnonzero(labels == BONAFIDE)
    spoof = np.flatnonzero(labels == SPOOF)
    n_spoof = min(len(spoof), int(round(spoof_per_bonafide * len(bonafide))))
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([bonafide, rng.choice(spoof, n_spoof, replace=False)]))


def choose_held_out_attacks(attacks, n_held_out, seed=42):
    spoof_attacks = sorted({a for a in attacks if a != "bonafide"})
    n = max(1, min(n_held_out, len(spoof_attacks) - 1))  # leave at least one attack for training
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(spoof_attacks, size=n, replace=False).tolist())


def drop_attacks(indices, attacks, held_out):
    """Train/val side of the attack-disjoint protocol: bonafide plus spoofs from non-held-out attacks."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] not in held]


def keep_only_attacks(indices, attacks, held_out):
    """Test side of the attack-disjoint protocol: bonafide plus spoofs from held-out attacks only."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] in held]


class ZipAudioArchive:
    """Members of a zip that may be remote (hf://, https://), read without downloading it whole.

    One zip filesystem is opened and reused, so the central directory of a multi-GB archive is read
    once and each clip is a single range read -- reopening a chained `zip://...::hf://...` URL per clip
    (what Phase 2's loader effectively did) re-reads the directory every time.
    """

    def __init__(self, archive, protocol="hf"):
        import fsspec

        self.archive, self.protocol = archive, protocol
        self.fs = fsspec.filesystem("zip", fo=archive, target_protocol=protocol)

    def members(self):
        return sorted(self.fs.find(""))

    def read_bytes(self, member):
        with self.fs.open(member, "rb") as handle:
            return handle.read()


def read_archive_metadata(archive):
    """Find the label metadata inside the archive (In-the-Wild ships `meta.csv` with file/label
    columns) rather than assuming a HuggingFace column schema that isn't documented."""
    import pandas as pd

    members = archive.members()
    csvs = [m for m in members if m.lower().endswith(".csv")]
    if not csvs:
        raise RuntimeError(f"no metadata .csv among {len(members)} archive members")
    meta_member = next((m for m in csvs if m.rsplit("/", 1)[-1].lower() == "meta.csv"), csvs[0])
    df = pd.read_csv(io.BytesIO(archive.read_bytes(meta_member)))

    def looks_like_files(col):
        return df[col].astype(str).str.lower().str.endswith((".wav", ".flac", ".mp3")).mean() > 0.9

    def looks_like_labels(col):
        values = set(df[col].astype(str).str.strip().str.lower().unique())
        return bool(values) and values <= (_BONAFIDE_WORDS | _SPOOF_WORDS)

    file_col = next((c for c in df.columns if looks_like_files(c)), None)
    label_col = next((c for c in df.columns if looks_like_labels(c)), None)
    if file_col is None or label_col is None:
        raise RuntimeError(f"could not identify file/label columns in {meta_member}: {list(df.columns)}")

    prefix = meta_member.rsplit("/", 1)[0] + "/" if "/" in meta_member else ""
    raw = df[label_col].astype(str).str.strip()
    table = pd.DataFrame({
        "member": prefix + df[file_col].astype(str),
        "raw_label": raw,
        "label": np.where(raw.str.lower().isin(_BONAFIDE_WORDS), BONAFIDE, SPOOF),
    })
    info = {"metadata_member": meta_member, "file_column": file_col, "label_column": label_col,
            "raw_label_values": sorted(raw.unique().tolist()), "rows": int(len(table)),
            "rows_bonafide": int((table.label == BONAFIDE).sum()), "rows_spoof": int((table.label == SPOOF).sum())}
    return table, info


def load_archive_clips_balanced(archive, n_per_class, seed=42, max_failures=None):
    """`n_per_class` clips of each class, read one by one; every failure is recorded, not hidden."""
    import soundfile as sf

    table, info = read_archive_metadata(archive)
    max_failures = max_failures if max_failures is not None else 2 * n_per_class
    rng = np.random.default_rng(seed)
    clips, failures = [], []
    for label in (BONAFIDE, SPOOF):
        pool = table[table.label == label].reset_index(drop=True)
        loaded = 0
        for pos in rng.permutation(len(pool)):
            if loaded >= n_per_class:
                break
            row = pool.iloc[int(pos)]
            try:
                wave, sr = sf.read(io.BytesIO(archive.read_bytes(row.member)), dtype="float32", always_2d=False)
            except Exception as exc:  # noqa: BLE001 -- recorded per clip, and capped below
                failures.append({"member": row.member, "error": f"{type(exc).__name__}: {exc}"})
                if len(failures) > max_failures:
                    raise RuntimeError(f"{len(failures)} clips failed to load; first: {failures[0]}") from exc
                continue
            if wave.ndim > 1:
                wave = wave.mean(axis=1)
            clips.append({"waveform": np.ascontiguousarray(wave, dtype=np.float32), "sample_rate": int(sr),
                          "label": label, "raw_label": row.raw_label, "speaker_id": "unknown",
                          "utterance_id": row.member, "attack": "bonafide" if label == BONAFIDE else "unknown"})
            loaded += 1
    report = {**info, "requested_per_class": n_per_class, "loaded": len(clips),
              "loaded_bonafide": sum(c["label"] == BONAFIDE for c in clips),
              "loaded_spoof": sum(c["label"] == SPOOF for c in clips),
              "failed": len(failures), "failures": failures[:20]}
    return clips, report


def write_synthetic_archive(path, n_real=4, n_fake=5, include_broken=True, sample_rate=TARGET_SR):
    """In-the-Wild-shaped zip (release_in_the_wild/*.wav + meta.csv with file/speaker/label) for the
    local smoke test and unit tests -- exercises the same fsspec zip path as the real hf:// archive.
    `include_broken` adds one undecodable member so failure recording is exercised too."""
    import soundfile as sf
    import zipfile

    rows = ["file,speaker,label"]
    rng = np.random.default_rng(0)
    with zipfile.ZipFile(path, "w") as zf:
        for i in range(n_real + n_fake):
            label = "bona-fide" if i < n_real else "spoof"
            t = np.arange(int(1.5 * sample_rate)) / sample_rate
            wave = (0.2 * np.sin(2 * np.pi * rng.uniform(100, 300) * t)).astype(np.float32)
            buf = io.BytesIO()
            sf.write(buf, wave, sample_rate, format="WAV")
            zf.writestr(f"release_in_the_wild/{i}.wav", buf.getvalue())
            rows.append(f"{i}.wav,Speaker {i % 3},{label}")
        if include_broken:
            zf.writestr("release_in_the_wild/broken.wav", b"not audio")
            rows.append("broken.wav,Speaker 0,spoof")
        zf.writestr("release_in_the_wild/meta.csv", "\n".join(rows))
    return str(path)


def write_synthetic_protocol(clips, path):
    """Protocol file in the official format, for the smoke test's protocol-verification path."""
    with open(path, "w", encoding="utf-8") as handle:
        for clip in clips:
            attack = "-" if clip["label"] == BONAFIDE else clip["attack"]
            key = "bonafide" if clip["label"] == BONAFIDE else "spoof"
            handle.write(f"{clip['speaker_id']} {clip['utterance_id']} - {attack} {key}\n")
    return str(path)


def make_synthetic_asvspoof(n_clips=120, seed=0, sample_rate=TARGET_SR, n_attacks=6, bonafide_pad_s=(0.02, 0.1)):
    """Tiny ASVspoof-shaped set for local smoke tests only -- never used for reported numbers.

    Deliberately includes the failure mode the real data is suspected of: spoof clips get more
    leading/trailing silence than bonafide ones, so the silence-leakage check has something to find.
    Spoof clips get one of `n_attacks` synthetic attack IDs, drawn independently of speaker (as in real
    ASVspoof, where every target speaker is spoofed by every system), from a separate RNG so adding
    attacks didn't change the synthetic audio other tests rely on.
    """
    rng = np.random.default_rng(seed)
    attack_rng = np.random.default_rng(seed + 1)
    clips = []
    for i in range(n_clips):
        label = BONAFIDE if i % 4 == 0 else SPOOF  # ~25% bonafide, imbalanced like LA
        attack = "bonafide" if label == BONAFIDE else f"A{7 + int(attack_rng.integers(n_attacks)):02d}"
        speech_s = rng.uniform(1.0, 2.5)
        t = np.arange(int(speech_s * sample_rate), dtype=np.float32) / sample_rate
        f0 = rng.uniform(100, 200)
        speech = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.1 * np.sin(2 * np.pi * 2 * f0 * t)
        if label == SPOOF:  # a crude high-frequency "vocoder artifact"
            speech += 0.05 * np.sin(2 * np.pi * 6500 * t)
        speech += 0.01 * rng.standard_normal(speech.shape).astype(np.float32)
        pad_s = rng.uniform(0.4, 0.8) if label == SPOOF else rng.uniform(*bonafide_pad_s)
        pad = 1e-4 * rng.standard_normal(int(pad_s * sample_rate)).astype(np.float32)
        clips.append({
            "waveform": np.concatenate([pad, speech, pad]).astype(np.float32),
            "sample_rate": sample_rate,
            "label": label,
            "speaker_id": f"SPK_{i % 12:02d}",
            "utterance_id": f"SYN_{i:04d}",
            "path": f"synthetic/{i}.wav",
            "attack": attack,
        })
    return clips


def class_counts(clips):
    labels = np.array([c["label"] for c in clips])
    return {"bonafide": int((labels == BONAFIDE).sum()), "spoof": int((labels == SPOOF).sum()),
            "total": int(len(labels))}


def pos_weight(clips):
    """Weight for the positive (bonafide, minority) class in BCE, as in the graph engine."""
    counts = class_counts(clips)
    return float(counts["spoof"] / max(counts["bonafide"], 1))


def _both_classes_present(clips, indices):
    labels = {clips[i]["label"] for i in indices}
    return BONAFIDE in labels and SPOOF in labels


def speaker_disjoint_split(clips, test_fraction=0.3, seed=42, attempts=12):
    """Split so no speaker appears in both train and test.

    A random clip-level split lets the same speaker land on both sides, and a model can then key on
    speaker identity/recording channel instead of spoofing artifacts -- the audio analogue of the
    Phase 1 random-vs-temporal split leak. Speakers are shuffled and sent whole to the test side
    until it reaches `test_fraction`, so a speaker is never split across sides.

    Caveat this does NOT fix: every clip here comes from the LA *evaluation* partition, so the same
    spoofing systems (A07-A19) appear on both sides. Unseen-attack generalization is not measured.
    """
    by_speaker = {}
    for idx, clip in enumerate(clips):
        by_speaker.setdefault(clip["speaker_id"], []).append(idx)

    target_test = test_fraction * len(clips)
    for attempt in range(attempts):
        rng = np.random.default_rng(seed + attempt)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)

        train_idx, test_idx, n_test = [], [], 0
        for speaker in speakers:
            members = by_speaker[speaker]
            # Send speakers to test until adding another would overshoot the target by more than it undershoots.
            if n_test + len(members) / 2 <= target_test:
                test_idx.extend(members)
                n_test += len(members)
            else:
                train_idx.extend(members)
        # Both sides need both classes, or EER is undefined on that side. Speakers are assigned whole,
        # so this isn't guaranteed by construction -- reshuffle and retry.
        if train_idx and test_idx and _both_classes_present(clips, train_idx) and _both_classes_present(clips, test_idx):
            return sorted(train_idx), sorted(test_idx)

    # Too few speakers (or a speaker holds an entire class) for a disjoint split with both classes on
    # both sides. Fall back to a random split rather than returning an unusable one; the caller's
    # split report will show shared_speakers > 0, making the fallback visible instead of silent.
    return random_split(clips, test_fraction, seed)


def random_split(clips, test_fraction=0.3, seed=42):
    """Clip-level random split -- speakers appear on both sides. Kept as the leakage comparison."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(clips))
    n_test = int(round(test_fraction * len(clips)))
    return sorted(order[n_test:].tolist()), sorted(order[:n_test].tolist())


def split_report(clips, train_idx, test_idx):
    train_speakers = {clips[i]["speaker_id"] for i in train_idx}
    test_speakers = {clips[i]["speaker_id"] for i in test_idx}
    return {
        "train": class_counts([clips[i] for i in train_idx]),
        "test": class_counts([clips[i] for i in test_idx]),
        "train_speakers": len(train_speakers),
        "test_speakers": len(test_speakers),
        "shared_speakers": len(train_speakers & test_speakers),
    }

In [ ]:
# ---- inlined from engines/voice/preprocess.py (edit the module, then rebuild) ----
"""Audio front-end: resample -> silence trim -> fixed length -> LFCC (+deltas), plus the silence
ablations and training-time augmentation used in Phase 2B.

**Why LFCC, not mel.** LFCC is the front-end of the official ASVspoof 2019/2021 baselines. Its
filters are spaced linearly, so it keeps resolution at high frequencies, which is where vocoder and
TTS artifacts live. Mel spacing compresses exactly that region because it is tuned to human
perception, which is the right trade for ASR and the wrong one for spoof detection.

LFCC is implemented here in plain torch (linear triangular filterbank -> log -> DCT-II -> deltas)
rather than via `torchaudio.transforms.LFCC`, so the pipeline has no torchaudio dependency.
`LFCCExtractor` is batched and runs on any device; `VoicePreprocessor` uses the same extractor for
single clips, so cached features and on-the-fly (augmented) features are the same computation.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F

TARGET_SR = 16000


def resample_linear(waveform, sr, target_sr=TARGET_SR):
    """Linear-interpolation resample. ASVspoof LA is already 16 kHz, so this is normally a no-op;
    it exists so an odd-rate out-of-domain clip doesn't silently go through at the wrong rate."""
    if sr == target_sr:
        return waveform, sr
    n_out = int(round(len(waveform) * target_sr / sr))
    x_in = np.linspace(0.0, 1.0, num=len(waveform), endpoint=False)
    x_out = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
    return np.interp(x_out, x_in, waveform).astype(np.float32), target_sr


def frame_energy_db(waveform, sr, frame_ms=20.0, hop_ms=10.0):
    frame, hop = int(sr * frame_ms / 1000), int(sr * hop_ms / 1000)
    if len(waveform) < frame:
        waveform = np.pad(waveform, (0, frame - len(waveform)))
    frames = np.lib.stride_tricks.sliding_window_view(waveform, frame)[::hop]
    rms = np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1) + 1e-12)
    return 20.0 * np.log10(rms + 1e-12), frame, hop


def trim_silence(waveform, sr, top_db=30.0, frame_ms=20.0, hop_ms=10.0):
    """Energy-based leading/trailing silence trim (librosa.effects.trim-style, no dependency).

    Returns (trimmed waveform, stats). Stats carry the leading/trailing silence and speech durations
    (what the silence-leakage probe measures) and the speech boundaries in samples (what the silence
    ablations cut at).
    """
    db, frame, hop = frame_energy_db(waveform, sr, frame_ms, hop_ms)
    voiced = np.flatnonzero(db > db.max() - top_db)
    total_s = len(waveform) / sr
    if len(voiced) == 0:  # nothing above the threshold: keep the clip, report it as all silence
        return waveform, {"total_s": total_s, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                          "speech_s": 0.0, "silence_ratio": 1.0, "all_silence": True,
                          "start_sample": 0, "end_sample": len(waveform)}
    start = int(voiced[0] * hop)
    end = min(len(waveform), int(voiced[-1] * hop + frame))
    trimmed = waveform[start:end]
    lead_s, trail_s = start / sr, (len(waveform) - end) / sr
    return trimmed, {
        "total_s": total_s,
        "lead_silence_s": lead_s,
        "trail_silence_s": trail_s,
        "speech_s": len(trimmed) / sr,
        "silence_ratio": (lead_s + trail_s) / total_s if total_s else 0.0,
        "all_silence": False,
        "start_sample": start,
        "end_sample": end,
    }


def fix_length(waveform, target_len):
    """Crop or repeat-pad to a fixed sample count (repeat-padding is the usual ASVspoof practice --
    zero-padding would inject exactly the silence cue this pipeline is trying to remove)."""
    if len(waveform) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(waveform) >= target_len:
        return waveform[:target_len]
    repeats = int(np.ceil(target_len / len(waveform)))
    return np.tile(waveform, repeats)[:target_len].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# Silence ablations (Phase 2B, Task 4). Each variant removes a different part of what the silence
# could be leaking, so comparing them says *which* part the untrimmed model was using:
#
#   variant                    silence duration   silence content (what's actually in it)
#   untrimmed                        kept              kept
#   zeros_original_duration          kept              removed (digital zeros)
#   zeros_fixed / noise_fixed        removed           removed (identical zeros / identical-level noise)
#   silence_only (fixed window)      removed           kept -- and nothing else is kept
#   trimmed                          removed           removed
# ---------------------------------------------------------------------------------------------------
SILENCE_ABLATIONS = ("zeros_fixed", "noise_fixed", "zeros_original_duration")


def ablate_silence(waveform, sr, mode, stats, fixed_seconds=0.5, noise_dbfs=-60.0, rng=None):
    """Rebuild a clip with its leading and trailing silence replaced according to `mode`.

    Both ends are replaced, not just the leading one: trailing silence also leaks the label (it is
    the second-strongest timing cue), so replacing only the lead would leave a known cue in place and
    confound the test.
    """
    start, end = stats["start_sample"], stats["end_sample"]
    speech = waveform[start:end]
    if mode == "zeros_fixed":
        pad = np.zeros(int(fixed_seconds * sr), dtype=np.float32)
        return np.concatenate([pad, speech, pad])
    if mode == "noise_fixed":
        n = int(fixed_seconds * sr)
        rng = rng if rng is not None else np.random.default_rng(0)
        noise = (rng.standard_normal(2 * n) * 10 ** (noise_dbfs / 20)).astype(np.float32)
        return np.concatenate([noise[:n], speech, noise[n:]])
    if mode == "zeros_original_duration":
        return np.concatenate([np.zeros(start, dtype=np.float32), speech,
                               np.zeros(len(waveform) - end, dtype=np.float32)])
    raise ValueError(f"unknown silence ablation: {mode}")


def silence_window(waveform, stats, n_samples):
    """Exactly `n_samples` of the clip's genuine silence (leading, then trailing), or None if the clip
    has less silence than that. A fixed-size window means silence *duration* cannot reach the model;
    only what the silence contains can."""
    start, end = stats["start_sample"], stats["end_sample"]
    silence = np.concatenate([waveform[:start], waveform[end:]])
    if len(silence) < n_samples:
        return None
    return silence[:n_samples].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# LFCC
# ---------------------------------------------------------------------------------------------------
def linear_filterbank(n_filters, n_fft, sr):
    """Triangular filters spaced linearly in Hz (the 'linear' in LFCC)."""
    edges = torch.linspace(0.0, sr / 2.0, n_filters + 2)
    bin_freqs = torch.linspace(0.0, sr / 2.0, n_fft // 2 + 1)
    bank = torch.zeros(n_filters, len(bin_freqs))
    for i in range(n_filters):
        low, center, high = edges[i], edges[i + 1], edges[i + 2]
        rising = (bin_freqs - low) / torch.clamp(center - low, min=1e-6)
        falling = (high - bin_freqs) / torch.clamp(high - center, min=1e-6)
        bank[i] = torch.clamp(torch.minimum(rising, falling), min=0.0)
    return bank


def dct_matrix(n_out, n_in):
    """Orthonormal DCT-II."""
    n = torch.arange(n_in, dtype=torch.float32)
    k = torch.arange(n_out, dtype=torch.float32).unsqueeze(1)
    mat = torch.cos(torch.pi * k * (2 * n + 1) / (2 * n_in)) * np.sqrt(2.0 / n_in)
    mat[0] = mat[0] / np.sqrt(2.0)
    return mat


def deltas(features, width=2):
    """Standard regression deltas over a +/-`width` frame window, edges replicated. Accepts
    [coeffs, frames] or [batch, coeffs, frames]."""
    squeeze = features.dim() == 2
    x = features.unsqueeze(0) if squeeze else features
    padded = F.pad(x, (width, width), mode="replicate")
    n_frames = x.shape[-1]
    out = torch.zeros_like(x)
    for n in range(1, width + 1):
        out += n * (padded[..., width + n: width + n + n_frames] - padded[..., width - n: width - n + n_frames])
    out = out / (2.0 * sum(n * n for n in range(1, width + 1)))
    return out.squeeze(0) if squeeze else out


class LFCCExtractor(torch.nn.Module):
    """Batched LFCC (+deltas): [batch, samples] -> [batch, n_coeffs*3, frames], on any device.

    `filterbank_augment`, if given, is applied to the log linear-filterbank energies before the DCT.
    That is where frequency masking means something (masking whole frequency bands); masking rows
    of the cepstrum afterwards would not correspond to any frequency region.
    """

    def __init__(self, sample_rate=TARGET_SR, n_fft=512, win_ms=20.0, hop_ms=10.0, n_filters=70,
                 n_coeffs=20, add_deltas=True):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = int(sample_rate * win_ms / 1000)
        self.hop_length = int(sample_rate * hop_ms / 1000)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.register_buffer("filterbank", linear_filterbank(n_filters, n_fft, sample_rate))
        self.register_buffer("dct", dct_matrix(n_coeffs, n_filters))
        self.register_buffer("window", torch.hann_window(self.win_length))

    @property
    def n_features(self):
        return self.n_coeffs * (3 if self.add_deltas else 1)

    def n_frames(self, n_samples):
        return n_samples // self.hop_length + 1

    def forward(self, waves, filterbank_augment=None):
        spec = torch.stft(waves, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length,
                          window=self.window, center=True, pad_mode="reflect", return_complex=True)
        log_fbank = torch.log(torch.matmul(self.filterbank, spec.abs() ** 2) + 1e-10)
        if filterbank_augment is not None:
            log_fbank = filterbank_augment(log_fbank)
        feats = torch.matmul(self.dct, log_fbank)
        if self.add_deltas:
            d1 = deltas(feats)
            feats = torch.cat([feats, d1, deltas(d1)], dim=1)
        return feats


class VoicePreprocessor:
    """waveform -> (n_coeffs*3, n_frames) LFCC+delta+delta-delta tensor.

    Defaults follow the ASVspoof LFCC baseline: 20 ms window / 10 ms hop, 70 linear filters,
    20 cepstral coefficients, plus first and second deltas = 60 feature dimensions.
    """

    def __init__(self, sample_rate=TARGET_SR, clip_seconds=4.0, n_fft=512, win_ms=20.0, hop_ms=10.0,
                 n_filters=70, n_coeffs=20, trim=True, top_db=30.0, add_deltas=True):
        self.sample_rate = sample_rate
        self.clip_seconds = clip_seconds
        self.clip_samples = int(sample_rate * clip_seconds)
        self.trim = trim
        self.top_db = top_db
        self.extractor = LFCCExtractor(sample_rate, n_fft, win_ms, hop_ms, n_filters, n_coeffs, add_deltas)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.hop_length = self.extractor.hop_length

    @property
    def n_features(self):
        return self.extractor.n_features

    @property
    def n_frames(self):
        return self.extractor.n_frames(self.clip_samples)

    def prepare_waveform(self, waveform, sr):
        waveform, sr = resample_linear(np.asarray(waveform, dtype=np.float32), sr, self.sample_rate)
        stats = {"total_s": len(waveform) / sr, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                 "speech_s": len(waveform) / sr, "silence_ratio": 0.0, "all_silence": False,
                 "start_sample": 0, "end_sample": len(waveform)}
        if self.trim:
            waveform, stats = trim_silence(waveform, sr, self.top_db)
        return fix_length(waveform, self.clip_samples), stats

    @torch.no_grad()
    def lfcc(self, waveform):
        wave = torch.as_tensor(waveform, dtype=torch.float32).unsqueeze(0)
        return self.extractor(wave)[0]

    def __call__(self, waveform, sr):
        fixed, stats = self.prepare_waveform(waveform, sr)
        return self.lfcc(fixed), stats

    def featurize_clips(self, clips, progress_every=1000):
        """Returns (features [N,1,F,T] float32, labels [N], per-clip silence stats)."""
        features, labels, stats = [], [], []
        for i, clip in enumerate(clips):
            feats, clip_stats = self(clip["waveform"], clip["sample_rate"])
            features.append(feats)
            labels.append(clip["label"])
            stats.append(clip_stats)
            if progress_every and (i + 1) % progress_every == 0:
                print(f"  featurized {i + 1}/{len(clips)} clips")
        x = torch.stack(features).unsqueeze(1)
        return x, torch.tensor(labels, dtype=torch.long), stats


# ---------------------------------------------------------------------------------------------------
# Training-time augmentation (Phase 2B Run B). Applied to training batches only -- never to
# validation or test, which the data sources enforce by only calling these when `training=True`.
# ---------------------------------------------------------------------------------------------------
def simulate_low_bitrate(waves, generator, downsample_factor=4, quant_bits=6, mu=255.0, p_apply=0.3):
    """Cheap proxy for a low-bitrate lossy-codec (e.g. MP3) round-trip: linear-interpolation
    downsample/upsample (the bandwidth loss a low bitrate causes) plus mu-law companding quantized to
    `quant_bits` (the quantization noise a low bitrate causes). This is NOT a real MP3 encode/decode --
    no differentiable, per-batch-fast MP3 codec exists -- it is a documented approximation of the two
    dominant artifacts a low-bitrate codec introduces, applied to a random subset of clips per batch.
    waves: [batch, samples] on any device, expected in [-1, 1]."""
    batch, n = waves.shape
    device = waves.device
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_apply).float()

    narrow = F.interpolate(waves.unsqueeze(1), size=max(n // downsample_factor, 1), mode="linear",
                           align_corners=False)
    wide = F.interpolate(narrow, size=n, mode="linear", align_corners=False).squeeze(1)

    x = wide.clamp(-1.0, 1.0)
    companded = torch.sign(x) * torch.log1p(mu * x.abs()) / math.log1p(mu)
    levels = 2 ** quant_bits - 1
    quantized = torch.round((companded + 1) / 2 * levels) / levels * 2 - 1
    expanded = torch.sign(quantized) * (torch.expm1(quantized.abs() * math.log1p(mu)) / mu)

    return waves + apply * (expanded - waves)


def augment_waveforms(waves, generator, snr_db=(10.0, 40.0), gain_db=(-6.0, 6.0), p_noise=0.5,
                      codec_sim=False):
    """Additive white noise at a random per-clip SNR (applied to a random half of clips) and a random
    per-clip gain. waves: [batch, samples] on any device. `codec_sim=True` (Phase 2C Run F only) adds
    `simulate_low_bitrate` on top -- Phase 2B explicitly skipped a codec augmentation for time."""
    batch = waves.shape[0]
    device = waves.device

    def uniform(lo, hi):
        return lo + (hi - lo) * torch.rand(batch, 1, generator=generator, device=device)

    signal_power = waves.pow(2).mean(dim=1, keepdim=True).clamp_min(1e-10)
    noise_power = signal_power / 10 ** (uniform(*snr_db) / 10)
    noise = torch.randn(waves.shape, generator=generator, device=device) * noise_power.sqrt()
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_noise).float()
    waves = waves + apply * noise
    waves = waves * 10 ** (uniform(*gain_db) / 20)
    if codec_sim:
        waves = simulate_low_bitrate(waves, generator)
    return waves


def spec_augment(generator, n_freq_masks=2, max_freq_width=8, n_time_masks=2, max_time_width=40):
    """SpecAugment-style masking for the log-filterbank tensor [batch, filters, frames]. Masked
    regions are set to each clip's mean log energy, so masking doesn't inject an artificial floor."""

    def apply(log_fbank):
        batch, n_filters, n_frames = log_fbank.shape
        device = log_fbank.device
        out = log_fbank.clone()
        fill = log_fbank.mean(dim=(1, 2), keepdim=True).expand_as(out)
        filters = torch.arange(n_filters, device=device).view(1, -1, 1)
        frames = torch.arange(n_frames, device=device).view(1, 1, -1)
        mask = torch.zeros_like(out, dtype=torch.bool)
        for _ in range(n_freq_masks):
            width = torch.randint(0, max_freq_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_filters - width).clamp_min(1)).long()
            mask |= (filters >= start) & (filters < start + width)
        for _ in range(n_time_masks):
            width = torch.randint(0, max_time_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_frames - width).clamp_min(1)).long()
            mask |= (frames >= start) & (frames < start + width)
        return torch.where(mask, fill, out)

    return apply

In [ ]:
# ---- inlined from engines/voice/evaluate.py (edit the module, then rebuild) ----
"""EER, DET curve and the silence-leakage probe.

Convention throughout: score = "how bonafide is this", label 1 = bonafide (the target class).
  FAR (false accept rate) = spoof clips scored above threshold  -> accepted as bonafide
  FRR (false reject rate)  = bonafide clips scored below threshold -> rejected as spoof
EER is where those two curves cross, found by linear interpolation between the two bracketing
operating points rather than by taking the nearest point, which is only an approximation.
"""
import numpy as np
from sklearn.metrics import roc_curve


def compute_eer(scores, labels, positive_label=1):
    """Equal Error Rate. Returns eer, the threshold at it, and FAR/FRR there."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    y = (labels == positive_label).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return {"eer": float("nan"), "threshold": float("nan"), "far": float("nan"),
                "frr": float("nan"), "note": "only one class present; EER undefined"}

    fpr, tpr, thresholds = roc_curve(y, scores)
    far, frr = fpr, 1.0 - tpr  # FAR rises with threshold-looseness, FRR falls
    diff = far - frr
    sign_change = np.flatnonzero(np.sign(diff[:-1]) * np.sign(diff[1:]) < 0)
    if len(sign_change) == 0:  # curves touch rather than cross (ties/small samples)
        idx = int(np.nanargmin(np.abs(diff)))
        return {"eer": float((far[idx] + frr[idx]) / 2), "threshold": float(thresholds[idx]),
                "far": float(far[idx]), "frr": float(frr[idx]), "interpolated": False}
    i = int(sign_change[0])
    # Linear interpolation of the crossing between operating points i and i+1.
    weight = diff[i] / (diff[i] - diff[i + 1])
    eer = float(far[i] + weight * (far[i + 1] - far[i]))
    frr_at = float(frr[i] + weight * (frr[i + 1] - frr[i]))
    threshold = float(thresholds[i] + weight * (thresholds[i + 1] - thresholds[i]))
    return {"eer": (eer + frr_at) / 2, "threshold": threshold, "far": eer, "frr": frr_at,
            "interpolated": True}


def det_curve_points(scores, labels, positive_label=1, max_points=512):
    """FAR/FRR pairs for plotting a DET curve, thinned to at most `max_points`."""
    y = (np.asarray(labels) == positive_label).astype(int)
    fpr, tpr, thresholds = roc_curve(y, np.asarray(scores, dtype=np.float64))
    far, frr = fpr, 1.0 - tpr
    if len(far) > max_points:
        keep = np.unique(np.linspace(0, len(far) - 1, max_points).astype(int))
        far, frr, thresholds = far[keep], frr[keep], thresholds[keep]
    return {"far": far.tolist(), "frr": frr.tolist(), "thresholds": thresholds.tolist()}


def per_class_accuracy(scores, labels, threshold, positive_label=1):
    scores, labels = np.asarray(scores), np.asarray(labels)
    predicted_positive = scores > threshold
    is_positive = labels == positive_label
    return {
        "threshold": float(threshold),
        "bonafide_accuracy": float(predicted_positive[is_positive].mean()) if is_positive.any() else None,
        "spoof_accuracy": float((~predicted_positive[~is_positive]).mean()) if (~is_positive).any() else None,
        "overall_accuracy": float((predicted_positive == is_positive).mean()),
        "n_bonafide": int(is_positive.sum()),
        "n_spoof": int((~is_positive).sum()),
    }


def _stratified_resample(labels, rng, positive_label=1):
    """Resample with replacement *within* each class, so every resample keeps the original class
    counts -- a plain resample could drop the (small) bonafide class and leave EER undefined."""
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    return np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])


def bootstrap_eer_ci(scores, labels, n_resamples=1000, seed=0, alpha=0.05, positive_label=1):
    """Percentile bootstrap CI for EER, stratified by class."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = _stratified_resample(labels, rng, positive_label)
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    samples = np.array(samples)
    return {
        "eer": compute_eer(scores, labels, positive_label)["eer"],
        "ci_low": float(np.quantile(samples, alpha / 2)),
        "ci_high": float(np.quantile(samples, 1 - alpha / 2)),
        "ci_width": float(np.quantile(samples, 1 - alpha / 2) - np.quantile(samples, alpha / 2)),
        "bootstrap_std": float(samples.std()),
        "n_resamples": n_resamples,
        "n_bonafide": int((labels == positive_label).sum()),
        "n_spoof": int((labels != positive_label).sum()),
    }


def bootstrap_ci_at_counts(scores, labels, n_bonafide, n_spoof, n_resamples=1000, seed=0, alpha=0.05,
                           positive_label=1):
    """CI width the same scores would have had with a smaller test set of the given class counts.
    Used to say how much the larger Phase 2B test set narrowed the CI versus Phase 2's, on the same
    model, instead of comparing CIs from different models."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = np.concatenate([rng.choice(pos, n_bonafide), rng.choice(neg, n_spoof)])
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    low, high = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
    return {"n_bonafide": int(n_bonafide), "n_spoof": int(n_spoof), "ci_low": float(low),
            "ci_high": float(high), "ci_width": float(high - low)}


def bootstrap_eer_difference(scores_a, labels_a, scores_b, labels_b, n_resamples=1000, seed=0,
                             alpha=0.05, paired=False, positive_label=1):
    """Percentile bootstrap CI for EER(a) - EER(b).

    paired=True: a and b are two models scored on the *same* clips in the same order, so each resample
    draws one set of indices used for both -- this cancels the test-set sampling noise the two share.
    paired=False: two different test sets (e.g. a random split vs. a speaker-disjoint split), each
    resampled independently.
    """
    sa, la = np.asarray(scores_a, dtype=np.float64), np.asarray(labels_a)
    sb, lb = np.asarray(scores_b, dtype=np.float64), np.asarray(labels_b)
    if paired and (len(sa) != len(sb) or not np.array_equal(la, lb)):
        raise ValueError("paired bootstrap needs both models scored on the same clips in the same order")
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        ia = _stratified_resample(la, rng, positive_label)
        ib = ia if paired else _stratified_resample(lb, rng, positive_label)
        diffs.append(compute_eer(sa[ia], la[ia], positive_label)["eer"] -
                     compute_eer(sb[ib], lb[ib], positive_label)["eer"])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = compute_eer(sa, la, positive_label)["eer"] - compute_eer(sb, lb, positive_label)["eer"]
    return {"difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "paired": paired, "n_resamples": n_resamples}


def speaker_leakage_call(diff):
    """Phase 2 printed "speaker leakage" for any positive difference, including +0.0009. The rule is
    now: only a CI that excludes zero counts. `diff` = EER(speaker-disjoint) - EER(random split), so a
    positive, significant difference means the random split flatters the model."""
    ci = f"{diff['difference']:+.4f} EER, 95% CI [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}]"
    if not diff["excludes_zero"]:
        return {"verdict": "not_detectable",
                "text": f"no leakage detectable at this sample size ({ci}; the CI includes zero)"}
    if diff["difference"] > 0:
        return {"verdict": "leakage_detected",
                "text": f"speaker leakage detected: the random split looks better ({ci}, CI excludes zero)"}
    return {"verdict": "random_split_worse",
            "text": f"the random split is significantly *worse* ({ci}) -- not leakage; investigate the splits"}


def per_attack_eer(scores, labels, attacks, positive_label=1):
    """EER of all bonafide clips against each spoofing attack separately."""
    scores, labels, attacks = np.asarray(scores), np.asarray(labels), np.asarray(attacks)
    bonafide = labels == positive_label
    out = {}
    for attack in sorted(set(attacks[~bonafide].tolist())):
        mask = bonafide | (attacks == attack)
        out[attack] = {"eer": compute_eer(scores[mask], labels[mask], positive_label)["eer"],
                       "n_spoof": int((attacks == attack).sum())}
    return out


def evaluate_scores(scores, labels, positive_label=1):
    eer = compute_eer(scores, labels, positive_label)
    return {
        **eer,
        "per_class_at_eer_threshold": per_class_accuracy(scores, labels, eer["threshold"], positive_label),
        "det_curve": det_curve_points(scores, labels, positive_label),
    }


def duration_leakage_probe(stats, labels, positive_label=1):
    """How much can clip timing alone tell you about the label?

    Scores each clip by a duration measure and computes the EER of that single feature. An EER far
    from 50% means the dataset's recording/segmentation artifacts leak the label, independent of any
    spoofing artifact -- the known ASVspoof silence-duration shortcut. Both directions are tried per
    measure, since which sign favours bonafide is itself an empirical question; the reported EER is
    the better (lower) of the two, i.e. the strongest cheat available from that measure.
    """
    labels = np.asarray(labels)
    out = {}
    for key in ("total_s", "lead_silence_s", "trail_silence_s", "speech_s", "silence_ratio"):
        values = np.array([s[key] for s in stats], dtype=np.float64)
        if np.allclose(values, values[0]):
            out[key] = {"eer": float("nan"), "note": "constant measure"}
            continue
        forward = compute_eer(values, labels, positive_label)["eer"]
        backward = compute_eer(-values, labels, positive_label)["eer"]
        better = min(forward, backward)
        y = (labels == positive_label).astype(float)
        out[key] = {
            "eer": float(better),
            "direction": "higher_is_bonafide" if forward <= backward else "lower_is_bonafide",
            "point_biserial_r": float(np.corrcoef(values, y)[0, 1]),
            "mean_bonafide": float(values[labels == positive_label].mean()),
            "mean_spoof": float(values[labels != positive_label].mean()),
        }
    return out


def leakage_verdict(probe, strong=0.35, mild=0.45):
    """Turn the probe into a stated finding. `strong`/`mild` are EER thresholds: the further below
    50% a single duration measure gets, the more label information timing alone carries."""
    usable = {k: v["eer"] for k, v in probe.items() if not np.isnan(v["eer"])}
    if not usable:
        return {"level": "unknown", "worst_measure": None, "worst_eer": None,
                "text": "No usable duration measure; leakage could not be assessed."}
    worst_measure = min(usable, key=usable.get)
    worst = usable[worst_measure]
    level = "strong" if worst < strong else "mild" if worst < mild else "none"
    text = {
        "strong": f"Timing alone separates the classes well ({worst_measure} EER {worst:.4f}), so this data "
                  f"carries the known ASVspoof silence-duration shortcut.",
        "mild": f"Timing alone is weakly predictive ({worst_measure} EER {worst:.4f}) -- some leakage, but far "
                f"from a free solution.",
        "none": f"No meaningful timing leakage: the best duration measure only reaches EER {worst:.4f} "
                f"({worst_measure}), close to the 0.5 of a coin flip.",
    }[level]
    return {"level": level, "worst_measure": worst_measure, "worst_eer": float(worst), "text": text}

In [ ]:
# ---- inlined from engines/voice/model.py (edit the module, then rebuild) ----
"""LCNN (Light CNN) spoof detector, the SSL layer-weighted head, and a shared training loop.

LCNN is the CNN architecture used by the ASVspoof baselines. Its defining piece is the
Max-Feature-Map (MFM) activation: a conv produces 2k channels, which are split in half and combined
with an element-wise max, halving the channels. That acts as a learned feature selector and keeps
the network small.

Every model outputs one logit per clip, where a higher score means "more bonafide" (label 1 =
bonafide), matching the EER convention in evaluate.py.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F



class MFM(torch.nn.Module):
    """Max-Feature-Map over channels (dim 1)."""

    def forward(self, x):
        a, b = torch.chunk(x, 2, dim=1)
        return torch.maximum(a, b)


def conv_mfm(in_ch, out_ch, kernel_size, padding):
    return torch.nn.Sequential(
        torch.nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding),
        MFM(),
    )


class LCNN(torch.nn.Module):
    MIN_SIZE = 16  # four 2x max-pools: each input axis must be at least 2**4 long

    def __init__(self, n_features=60, n_frames=401, channels=(16, 24, 32, 32), dropout=0.3, fc_dim=80):
        super().__init__()
        if n_features < self.MIN_SIZE or n_frames < self.MIN_SIZE:
            raise ValueError(f"LCNN needs at least {self.MIN_SIZE} feature rows and frames, got "
                             f"{n_features}x{n_frames} -- use a longer input window")
        c1, c2, c3, c4 = channels
        self.blocks = torch.nn.Sequential(
            conv_mfm(1, c1, 5, 2),
            torch.nn.MaxPool2d(2),
            conv_mfm(c1, c1, 1, 0),
            torch.nn.BatchNorm2d(c1),
            conv_mfm(c1, c2, 3, 1),
            torch.nn.MaxPool2d(2),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c2, 1, 0),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c3, 3, 1),
            torch.nn.MaxPool2d(2),
            conv_mfm(c3, c3, 1, 0),
            torch.nn.BatchNorm2d(c3),
            conv_mfm(c3, c4, 3, 1),
            torch.nn.MaxPool2d(2),
        )
        with torch.no_grad():  # measure the flattened size instead of deriving it from the pooling arithmetic
            flat_dim = self.blocks(torch.zeros(1, 1, n_features, n_frames)).numel()
        self.head = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(flat_dim, fc_dim * 2),
            MFM(),  # operates on dim 1, which is the feature dim for a 2D tensor
            torch.nn.BatchNorm1d(fc_dim),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(fc_dim, 1),
        )

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(-1)


class SSLHead(torch.nn.Module):
    """Classifier over pooled self-supervised encoder features [batch, layers, dim].

    A learned softmax weight per encoder layer mixes the layers into one vector, then a small MLP
    scores it. The layer weights are reported after training: they say which depth of the encoder
    carried the spoofing information.
    """

    def __init__(self, n_layers, feat_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.layer_logits = torch.nn.Parameter(torch.zeros(n_layers))
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(feat_dim),
            torch.nn.Linear(feat_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        weights = torch.softmax(self.layer_logits, dim=0)
        return self.mlp((weights.view(1, -1, 1) * x).sum(dim=1)).squeeze(-1)

    def layer_weights(self):
        return torch.softmax(self.layer_logits.detach(), dim=0).cpu().tolist()

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


class SSLSeqHead(torch.nn.Module):
    """Phase 2C Model E: a single-layer bidirectional LSTM over one frozen SSL layer's per-frame
    token sequence [batch, frames, hidden], mean-pooled over time, then a small MLP head.

    Only one SSL layer is used (picked from Model C's learned layer weights, logged in the notebook),
    so there is no per-layer weighting mechanism here -- unlike `SSLHead`, which mixes every layer.
    Kept deliberately small (BiLSTM hidden size 64 by default): the goal is to test whether *any*
    temporal modeling helps, not to add capacity as a confound. `n_added_params` lets the notebook log
    that confound directly against `SSLHead`'s parameter count.
    """

    def __init__(self, feat_dim, lstm_hidden=64, hidden=128, dropout=0.3):
        super().__init__()
        self.lstm = torch.nn.LSTM(feat_dim, lstm_hidden, num_layers=1, batch_first=True, bidirectional=True)
        out_dim = 2 * lstm_hidden
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(out_dim),
            torch.nn.Linear(out_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.mlp(out.mean(dim=1)).squeeze(-1)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def batches(n, batch_size, shuffle, generator=None):
    order = torch.randperm(n, generator=generator) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


def score_model(model, fetch, indices, batch_size=64):
    """Scores (higher = more bonafide) for `indices`, in order. `fetch(idx, training=False)`."""
    model.eval()
    indices = np.asarray(indices)
    out = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            out.append(model(fetch(indices[start:start + batch_size], False)).detach().float().cpu())
    return torch.cat(out).numpy().astype(np.float64) if out else np.zeros(0)


def train_model(model, fetch, labels, train_idx, pos_weight, device, val_idx=None, epochs=20,
                batch_size=32, lr=3e-4, weight_decay=1e-4, patience=5, seed=42, verbose=True):
    """Class-weighted BCE training with optional early stopping on validation EER.

    `fetch(idx, training)` returns the model input for clip indices `idx` on `device`; data sources
    apply augmentation only when `training` is True, so validation and test inputs are never
    augmented. `labels` is indexed by the same clip indices.
    """
    labels = torch.as_tensor(np.asarray(labels))
    train_idx = np.asarray(train_idx)
    generator = torch.Generator().manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_weight_t = torch.tensor([pos_weight], device=device, dtype=torch.float32)

    history, best = [], {"eer": math.inf, "epoch": 0, "state": None}
    since_improve, val_unusable = 0, False
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_seen = 0.0, 0
        for positions in batches(len(train_idx), batch_size, shuffle=True, generator=generator):
            if len(positions) < 2:
                continue  # BatchNorm needs at least 2 samples
            idx = train_idx[positions.numpy()]
            xb = fetch(idx, True)
            yb = labels[idx].to(device).float()
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(xb), yb, pos_weight=pos_weight_t)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)
            n_seen += len(idx)
        epoch_loss = total_loss / max(n_seen, 1)
        if not math.isfinite(epoch_loss):
            raise FloatingPointError(f"training diverged at epoch {epoch} (loss={epoch_loss})")

        entry = {"epoch": epoch, "train_loss": epoch_loss}
        if val_idx is not None and not val_unusable:
            entry["val_eer"] = compute_eer(score_model(model, fetch, val_idx), labels[np.asarray(val_idx)].numpy())["eer"]
            if not math.isfinite(entry["val_eer"]):
                # One-class validation set: EER is undefined, so model selection and early stopping
                # have nothing to act on. Say so loudly rather than silently keeping the last epoch.
                val_unusable = True
                print("  WARNING: validation EER is undefined (validation set has a single class) -- "
                      "disabling early stopping and best-epoch selection for this run")
            elif entry["val_eer"] < best["eer"] - 1e-6:
                best = {"eer": entry["val_eer"], "epoch": epoch,
                        "state": {k: v.detach().clone() for k, v in model.state_dict().items()}}
                since_improve = 0
            else:
                since_improve += 1
        history.append(entry)
        if verbose:
            print(f"  epoch {epoch:3d}  loss {epoch_loss:.4f}" +
                  (f"  val EER {entry['val_eer']:.4f}" if "val_eer" in entry else ""))
        if patience is not None and val_idx is not None and not val_unusable and since_improve >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (best val EER {best['eer']:.4f} at epoch {best['epoch']})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    usable_val = val_idx is not None and not val_unusable
    return {"history": history, "best_val_eer": best["eer"] if usable_val else None,
            "best_epoch": best["epoch"] if usable_val else len(history),
            "epochs_run": len(history), "validation_usable": usable_val}


def train_lcnn(model, x_train, y_train, pos_weight, device, epochs=20, batch_size=32, lr=3e-4,
               weight_decay=1e-4, x_val=None, y_val=None, patience=5, seed=42, verbose=True):
    """In-memory wrapper around `train_model` (the Phase 2 notebook's interface)."""
    n_train = len(x_train)
    x_all = x_train if x_val is None else torch.cat([x_train, x_val])
    y_all = y_train if y_val is None else torch.cat([y_train, y_val])
    val_idx = None if x_val is None else np.arange(n_train, n_train + len(x_val))
    return train_model(model, lambda idx, training: x_all[torch.as_tensor(idx)].to(device), y_all,
                       np.arange(n_train), pos_weight, device, val_idx=val_idx, epochs=epochs,
                       batch_size=batch_size, lr=lr, weight_decay=weight_decay, patience=patience,
                       seed=seed, verbose=verbose)


@torch.no_grad()
def predict_scores(model, x, device, batch_size=64):
    """Scores where higher = more bonafide, for an in-memory feature tensor."""
    return score_model(model, lambda idx, training: x[torch.as_tensor(idx)].to(device),
                       np.arange(len(x)), batch_size)

## 1. Data loading + EDA

In [ ]:
t0 = time.time()
if SMOKE_TEST:
    clips = make_synthetic_asvspoof(N_CLIPS)
    dataset_name = "synthetic (smoke test)"
else:
    print(f"streaming {N_CLIPS} clips from {ASVSPOOF_DATASET} ...")
    clips = load_asvspoof_subset(N_CLIPS, seed=SEED)
    dataset_name = f"{ASVSPOOF_DATASET} (split '{ASVSPOOF_SPLIT}' = LA evaluation partition)"
print(f"loaded {len(clips)} clips in {time.time() - t0:.0f}s")

durations = np.array([len(c["waveform"]) / c["sample_rate"] for c in clips])
sample_rates = sorted({c["sample_rate"] for c in clips})
speakers = sorted({c["speaker_id"] for c in clips})
counts = class_counts(clips)
weight_for_bonafide = pos_weight(clips)

eda = {
    "dataset": dataset_name,
    "clips_requested": N_CLIPS,
    "clips_loaded": len(clips),
    "bonafide": counts["bonafide"],
    "spoof": counts["spoof"],
    "bonafide_share": counts["bonafide"] / max(counts["total"], 1),
    "pos_weight_for_bonafide": weight_for_bonafide,
    "speakers": len(speakers),
    "sample_rates_hz": sample_rates,
    "duration_s": {"mean": float(durations.mean()), "min": float(durations.min()),
                   "max": float(durations.max()), "median": float(np.median(durations)),
                   "total_hours": float(durations.sum() / 3600)},
}
print(json.dumps(eda, indent=2))
save_progress("eda", eda)

In [ ]:
eda_md = f"""# Voice Data Exploration -- ASVspoof 2019 LA

Source: `{eda['dataset']}`, streamed with the HuggingFace `datasets` library.

**Subset actually used for every number in this phase: {eda['clips_loaded']} clips** (of the 71,237 in
the LA evaluation partition), sampled with a shuffle buffer at seed {SEED}. This is the figure to quote;
it is a subset, not the full partition.

## Class balance
- Bonafide: {eda['bonafide']}
- Spoof: {eda['spoof']}
- Bonafide share: {eda['bonafide_share']:.2%}

Spoof outnumbers bonafide by about {eda['pos_weight_for_bonafide']:.1f} to 1, so the training loss is
class-weighted by that ratio and the headline metric is EER rather than accuracy -- the same
reasoning that made PR-AUC the metric for the graph engine.

## Clip durations (as provided, before any trimming)
- Mean: {eda['duration_s']['mean']:.2f} s
- Median: {eda['duration_s']['median']:.2f} s
- Min: {eda['duration_s']['min']:.2f} s
- Max: {eda['duration_s']['max']:.2f} s
- Total audio: {eda['duration_s']['total_hours']:.2f} hours

## Sample rate
- As provided: {', '.join(str(s) for s in eda['sample_rates_hz'])} Hz (the pipeline resamples to 16 kHz if needed).

## Speakers
- Distinct speaker IDs: {eda['speakers']}

Speaker identity matters for the split: a random clip-level split would place the same speaker on
both sides, letting the model key on voice/channel identity instead of spoofing artifacts. The
headline split in this phase is speaker-disjoint; see `docs/design_decisions.md`.

## Known caveat
These clips all come from the LA **evaluation** partition, so the same spoofing systems (A07-A19)
appear in both the train and test halves of our split. Performance against *unseen* attacks is not
measured here.
"""
(DOCS_DIR / "voice_data_exploration.md").write_text(eda_md, encoding="utf-8")
print(f"wrote {DOCS_DIR / 'voice_data_exploration.md'}")

## 2. Preprocessing pipeline

One `VoicePreprocessor` builds every feature tensor used below (train, test, and the out-of-domain
clips in task 6), so those paths cannot silently diverge. Two variants are built: **trimmed**
(the real pipeline) and **untrimmed** (identical apart from the silence trim), which exists purely
to measure how much the model relies on silence -- task 3.

In [ ]:
t0 = time.time()
pre_trimmed = VoicePreprocessor(clip_seconds=CLIP_SECONDS, trim=True)
pre_untrimmed = VoicePreprocessor(clip_seconds=CLIP_SECONDS, trim=False)

x_trimmed, y_all, silence_stats = pre_trimmed.featurize_clips(clips)
x_untrimmed, _, _ = pre_untrimmed.featurize_clips(clips)
print(f"features {tuple(x_trimmed.shape)} (clips, channel, LFCC+deltas, frames) in {time.time() - t0:.0f}s")

feature_config = {
    "front_end": "LFCC + delta + delta-delta",
    "n_coeffs": pre_trimmed.n_coeffs,
    "n_features": pre_trimmed.n_features,
    "n_frames": pre_trimmed.n_frames,
    "clip_seconds": CLIP_SECONDS,
    "window_ms": 20.0,
    "hop_ms": 10.0,
    "n_linear_filters": 70,
    "trim_top_db": pre_trimmed.top_db,
    "why_lfcc": "LFCC is the official ASVspoof 2019/2021 baseline front-end; linear filter spacing "
                "keeps high-frequency resolution where vocoder/TTS artifacts live, which mel spacing "
                "compresses away.",
}
save_progress("feature_config", feature_config)

## 3. Silence-leakage check

ASVspoof 2019 LA is known to leak the label through silence duration: bonafide and spoof clips were
produced differently, so leading/trailing silence alone can separate them without any spoofing
artifact. The probe scores each clip by a timing measure alone and computes that measure's EER --
an EER far from 0.5 means timing alone is a shortcut. It is run **before** trimming and **after**,
and then confirmed by training the same model on untrimmed features (task 4).

In [ ]:
labels_np = y_all.numpy()
probe_before = duration_leakage_probe(silence_stats, labels_np)
verdict_before = leakage_verdict(probe_before)

# After trimming, the model sees a fixed-length window of trimmed speech: leading/trailing silence is
# gone by construction, so those measures become constant. What can still carry signal is how much
# speech remained, so that is what the "after" probe scores.
stats_after = [{"total_s": s["speech_s"], "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                "speech_s": s["speech_s"], "silence_ratio": 0.0} for s in silence_stats]
probe_after = duration_leakage_probe(stats_after, labels_np)
verdict_after = leakage_verdict(probe_after)

print("BEFORE trimming (EER of each timing measure used alone; 0.5 = no information):")
for key, value in probe_before.items():
    if not math.isnan(value["eer"]):
        print(f"  {key:18s} EER {value['eer']:.4f}  r={value['point_biserial_r']:+.3f}  "
              f"bonafide mean {value['mean_bonafide']:.3f}s vs spoof {value['mean_spoof']:.3f}s")
print(f"  -> {verdict_before['text']}")
print("\nAFTER trimming (leading/trailing silence removed; only residual speech duration remains):")
for key, value in probe_after.items():
    if not math.isnan(value["eer"]):
        print(f"  {key:18s} EER {value['eer']:.4f}  r={value['point_biserial_r']:+.3f}")
print(f"  -> {verdict_after['text']}")

leakage = {
    "before_trim": probe_before, "after_trim": probe_after,
    "verdict_before": verdict_before, "verdict_after": verdict_after,
    "worst_eer_before": verdict_before["worst_eer"], "worst_eer_after": verdict_after["worst_eer"],
    "note": "Every model input is a fixed-length window, so absolute clip duration cannot reach the "
            "model regardless; these probes measure what the raw data leaks, and the trimmed-vs-"
            "untrimmed models below measure whether the model actually exploited it.",
}
save_progress("silence_leakage", leakage)

## 4. LCNN training

Three runs of the same LCNN, differing only in one variable each, so each comparison is clean:
- `lcnn_trimmed_speaker_disjoint` -- **the headline model**.
- `lcnn_untrimmed_speaker_disjoint` -- same split, silence left in: does the model exploit silence?
- `lcnn_trimmed_random_split` -- same features, speakers allowed on both sides: how much does
  speaker leakage inflate the number?

In [ ]:
def sub_split(indices, fraction, seed, disjoint):
    """Split a subset of clip indices, returning global indices."""
    subset = [clips[i] for i in indices]
    splitter = speaker_disjoint_split if disjoint else random_split
    left, right = splitter(subset, fraction, seed)
    return [indices[i] for i in left], [indices[i] for i in right]


SPLITS = {}
for split_name, disjoint in [("speaker_disjoint", True), ("random", False)]:
    trainval_idx, test_idx = (speaker_disjoint_split if disjoint else random_split)(clips, TEST_FRACTION, SEED)
    train_idx, val_idx = sub_split(trainval_idx, VAL_FRACTION, SEED + 1, disjoint)
    SPLITS[split_name] = {"train": train_idx, "val": val_idx, "test": test_idx,
                          "report": split_report(clips, train_idx, test_idx),
                          "val_counts": class_counts([clips[i] for i in val_idx])}
    report = SPLITS[split_name]["report"]
    print(f"{split_name:16s} train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)} | "
          f"test bonafide {report['test']['bonafide']} spoof {report['test']['spoof']} | "
          f"val bonafide {SPLITS[split_name]['val_counts']['bonafide']} | "
          f"speakers shared between train and test: {report['shared_speakers']}")
    if report["shared_speakers"] and split_name == "speaker_disjoint":
        print("  WARNING: speaker-disjoint split fell back to a random split (too few speakers to "
              "keep both classes on both sides) -- the headline number is NOT speaker-disjoint")
save_progress("splits", {k: {**v["report"], "n_train": len(v["train"]), "n_val": len(v["val"]),
                             "n_test": len(v["test"]), "val_counts": v["val_counts"]}
                         for k, v in SPLITS.items()})

In [ ]:
RUNS = [
    ("lcnn_trimmed_speaker_disjoint", "trimmed", "speaker_disjoint", "headline model"),
    ("lcnn_untrimmed_speaker_disjoint", "untrimmed", "speaker_disjoint", "silence-leakage comparison"),
    ("lcnn_trimmed_random_split", "trimmed", "random", "speaker-leakage comparison"),
]
FEATURES = {"trimmed": x_trimmed, "untrimmed": x_untrimmed}
MODEL_RESULTS, SCORES, MODELS = {}, {}, {}

for name, feature_set, split_name, role in RUNS:
    print(f"\n=== {name} ({role}) ===")
    x, split = FEATURES[feature_set], SPLITS[split_name]
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
    train_clips = [clips[i] for i in train_idx]

    torch.manual_seed(SEED)
    model = LCNN(n_features=pre_trimmed.n_features, n_frames=pre_trimmed.n_frames)
    t0 = time.time()
    fit = train_lcnn(model, x[train_idx], y_all[train_idx], pos_weight(train_clips), DEVICE,
                     epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE,
                     x_val=x[val_idx], y_val=y_all[val_idx], patience=PATIENCE, seed=SEED)
    scores = predict_scores(model, x[test_idx], DEVICE)
    report = evaluate_scores(scores, y_all[test_idx].numpy())

    MODEL_RESULTS[name] = {
        "features": feature_set, "split": split_name, "role": role,
        "eer": report["eer"], "eer_threshold": report["threshold"],
        "per_class_at_eer_threshold": report["per_class_at_eer_threshold"],
        "best_val_eer": fit["best_val_eer"], "best_epoch": fit["best_epoch"],
        "validation_usable": fit["validation_usable"], "epochs_run": fit["epochs_run"],
        "train_seconds": round(time.time() - t0, 1),
        "train_loss_first_last": [fit["history"][0]["train_loss"], fit["history"][-1]["train_loss"]],
    }
    SCORES[name] = {"scores": scores, "labels": y_all[test_idx].numpy(), "det": report["det_curve"]}
    MODELS[name] = model
    accuracy = report["per_class_at_eer_threshold"]
    print(f"  test EER {report['eer']:.4f} | bonafide acc {accuracy['bonafide_accuracy']:.4f} | "
          f"spoof acc {accuracy['spoof_accuracy']:.4f} | {fit['epochs_run']} epochs, "
          f"{MODEL_RESULTS[name]['train_seconds']:.0f}s")
    save_progress("models", MODEL_RESULTS)

## 5. Evaluation: EER, DET curve, per-class accuracy

In [ ]:
headline = "lcnn_trimmed_speaker_disjoint"
headline_eer = MODEL_RESULTS[headline]["eer"]
silence_effect = MODEL_RESULTS["lcnn_untrimmed_speaker_disjoint"]["eer"] - headline_eer
speaker_effect = headline_eer - MODEL_RESULTS["lcnn_trimmed_random_split"]["eer"]
# The two splits have different test sets, so this is an unpaired bootstrap: only a CI that excludes
# zero is reported as leakage (Phase 2 labelled a +0.0009 difference "speaker leakage" on sign alone).
speaker_leakage = bootstrap_eer_difference(
    SCORES[headline]["scores"], SCORES[headline]["labels"],
    SCORES["lcnn_trimmed_random_split"]["scores"], SCORES["lcnn_trimmed_random_split"]["labels"],
    n_resamples=N_BOOTSTRAP, seed=SEED)
speaker_call = speaker_leakage_call(speaker_leakage)

print(f"{'model':34s} {'EER':>8s} {'bonafide acc':>13s} {'spoof acc':>10s}")
for name, result in MODEL_RESULTS.items():
    accuracy = result["per_class_at_eer_threshold"]
    print(f"{name:34s} {result['eer']:8.4f} {accuracy['bonafide_accuracy']:13.4f} "
          f"{accuracy['spoof_accuracy']:10.4f}")
print(f"\nheadline EER (trimmed features, speaker-disjoint split): {headline_eer:.4f}")
print(f"untrimmed minus trimmed EER: {silence_effect:+.4f} "
      f"({'silence helped the model' if silence_effect < 0 else 'silence did not help the model'})")
print(f"speaker-disjoint minus random-split EER: {speaker_call['text']}")

import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 5))
for name in MODEL_RESULTS:
    det = SCORES[name]["det"]
    plt.plot(np.array(det["far"]) * 100, np.array(det["frr"]) * 100,
             label=f"{name} (EER {MODEL_RESULTS[name]['eer']:.3f})")
plt.plot([0, 100], [0, 100], "k--", linewidth=0.8, label="EER line")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("False accept rate (%)  -- spoof accepted as bonafide")
plt.ylabel("False reject rate (%)  -- bonafide rejected")
plt.title("DET curves, ASVspoof 2019 LA subset")
plt.legend(fontsize=7)
plt.grid(True, which="both", linewidth=0.3)
plt.tight_layout()
det_path = OUTPUT_DIR / "det_curves.png"
plt.savefig(det_path, dpi=120)
plt.show()
print(f"wrote {det_path}")

save_progress("evaluation", {
    "headline_model": headline,
    "headline_eer": headline_eer,
    "untrimmed_minus_trimmed_eer": silence_effect,
    "speaker_disjoint_minus_random_eer": speaker_effect,
    "speaker_leakage_bootstrap": speaker_leakage,
    "speaker_leakage_verdict": speaker_call,
    "det_curves": {name: SCORES[name]["det"] for name in MODEL_RESULTS},
})

## 6. Cross-dataset spot-check (stretch goal)

A handful of In-the-Wild clips, which come from real recordings rather than ASVspoof's studio
conditions. With this few clips the numbers are indicative only, never a headline. Non-fatal: if the
dataset can't be reached or its labels can't be interpreted, that is recorded and the notebook
continues.

In [ ]:
ood = {"status": "not_run", "n_clips": 0}
try:
    if SMOKE_TEST:
        ood_clips = make_synthetic_asvspoof(N_OOD_CLIPS, seed=7)
        ood_source = "synthetic (smoke test)"
    else:
        print(f"streaming {N_OOD_CLIPS} clips from {IN_THE_WILD_DATASET} ...")
        ood_clips = load_in_the_wild_clips(N_OOD_CLIPS, seed=SEED)
        ood_source = IN_THE_WILD_DATASET

    x_ood, _, _ = pre_trimmed.featurize_clips(ood_clips, progress_every=None)
    ood_scores = predict_scores(MODELS[headline], x_ood, DEVICE)
    ood_labels = [c.get("label") for c in ood_clips]
    labelled = [(s, l) for s, l in zip(ood_scores, ood_labels) if l is not None]

    ood = {
        "status": "ok",
        "source": ood_source,
        "n_clips": len(ood_clips),
        "raw_label_values": sorted({str(c.get("raw_label")) for c in ood_clips}) if not SMOKE_TEST else ["synthetic"],
        "label_column": ood_clips[0].get("label_column") if not SMOKE_TEST else "label",
        "scores": [float(s) for s in ood_scores],
        "labels": [None if l is None else int(l) for l in ood_labels],
        "mean_score_bonafide": float(np.mean([s for s, l in labelled if l == BONAFIDE])) if any(l == BONAFIDE for _, l in labelled) else None,
        "mean_score_spoof": float(np.mean([s for s, l in labelled if l == SPOOF])) if any(l == SPOOF for _, l in labelled) else None,
    }
    if len({l for _, l in labelled}) == 2:
        ood["eer"] = compute_eer(np.array([s for s, _ in labelled]), np.array([l for _, l in labelled]))["eer"]
        ood["eer_caveat"] = (f"EER over only {len(labelled)} clips -- indicative, not a reportable "
                             f"operating point (in-domain EER used {len(SPLITS['speaker_disjoint']['test'])} clips).")
    else:
        ood["eer"] = None
        ood["eer_caveat"] = "Only one class present among the sampled clips; EER undefined."
    print(json.dumps({k: v for k, v in ood.items() if k != "scores"}, indent=2))
except Exception as exc:  # noqa: BLE001 -- a stretch goal must not sink the run
    ood = {"status": "failed", "error": f"{type(exc).__name__}: {exc}",
           "note": "Cross-dataset spot-check skipped; in-domain results above are unaffected."}
    print(f"cross-dataset spot-check failed, continuing: {ood['error']}")
save_progress("cross_dataset_spot_check", ood)

## 7. Closing analysis and save

In [ ]:
findings = [
    f"Headline: an LCNN on LFCC features reaches {headline_eer:.4f} EER on a held-out, speaker-disjoint "
    f"test split of {len(SPLITS['speaker_disjoint']['test'])} clips drawn from a {len(clips)}-clip subset "
    f"of the ASVspoof 2019 LA evaluation partition.",
    f"Silence leakage: before trimming, the most predictive timing measure alone reached EER "
    f"{leakage['worst_eer_before']:.4f} ({verdict_before['worst_measure']}); after trimming, "
    f"{leakage['worst_eer_after']:.4f}. {verdict_before['text']}",
]
if silence_effect < -0.01:
    findings.append(f"Training on untrimmed features scored {abs(silence_effect):.4f} EER *better* "
                    f"({MODEL_RESULTS['lcnn_untrimmed_speaker_disjoint']['eer']:.4f}), i.e. the model does "
                    f"exploit silence when it is left in -- trimming removes a real shortcut rather than a "
                    f"hypothetical one.")
elif silence_effect > 0.01:
    findings.append(f"Training on untrimmed features scored {silence_effect:.4f} EER *worse* "
                    f"({MODEL_RESULTS['lcnn_untrimmed_speaker_disjoint']['eer']:.4f}), so the leftover silence "
                    f"was noise for this model, not a usable shortcut.")
else:
    findings.append(f"Trimmed and untrimmed models land within {abs(silence_effect):.4f} EER of each other, "
                    f"so the silence cue was not decisive for this model either way.")
findings.append(f"Speaker leakage (random-split EER {MODEL_RESULTS['lcnn_trimmed_random_split']['eer']:.4f} vs "
                f"speaker-disjoint {headline_eer:.4f}): {speaker_call['text']}.")
if ood.get("status") == "ok" and ood.get("eer") is not None:
    findings.append(f"Out of domain: on {ood['n_clips']} In-the-Wild clips the same model scores EER "
                    f"{ood['eer']:.4f}. {ood['eer_caveat']}")
elif ood.get("status") == "ok":
    findings.append(f"Out of domain: {ood['n_clips']} In-the-Wild clips scored, but {ood['eer_caveat'][0].lower()}"
                    f"{ood['eer_caveat'][1:]}")
else:
    findings.append("Out of domain: the spot-check did not run, so nothing is claimed about generalization.")
findings.append("Scope limit: every clip comes from the LA evaluation partition, so the same spoofing "
                "systems appear in train and test. Performance against unseen attack types is not measured.")

closing_paragraph = " ".join(findings)
print(closing_paragraph)
save_progress("closing_paragraph", closing_paragraph)

for name, model in MODELS.items():
    torch.save(model.state_dict(), OUTPUT_DIR / f"{name}.pt")
    print(f"wrote {OUTPUT_DIR / f'{name}.pt'}")
np.savez_compressed(OUTPUT_DIR / "test_scores.npz",
                    **{f"{name}__{field}": SCORES[name][field] for name in SCORES for field in ("scores", "labels")})
print(f"wrote {OUTPUT_DIR / 'test_scores.npz'}")
print(f"\nFull results: {RESULTS_PATH}")
print(json.dumps({k: v for k, v in RESULTS.items() if k != "evaluation"}, indent=2, default=float))